# System Dynamics Simulator for Net Landfill Load Forecasting

Research project: **Sustainable Waste Management System Based on Integrated Risk Assessment and System Dynamics for Net Landfill Load Forecasting at Butus Landfill**.

This notebook runs a daily discrete-time System Dynamics simulation for residual waste, Net Landfill Load, accumulated waste, remaining capacity, overload accumulation, time to overload, scenario comparison, and sensitivity analysis. Fuzzy AHP risk parameters are used as a risk-informed interpretation layer, not as direct drivers of Net Landfill Load.


## 2. Import libraries


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import nbformat
except ImportError:
    nbformat = None

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.7,
})


## 3. Define project paths


In [ ]:
def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "01_data" / "parameters").exists() and (candidate / "src").exists():
            return candidate
    for candidate in candidates:
        nested = candidate / "System_Dynamics_Simulator"
        if (nested / "01_data" / "parameters").exists() and (nested / "src").exists():
            return nested
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from inside the "
        "System_Dynamics_Simulator folder or upload the full project folder."
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "01_data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
PARAMETER_DIR = DATA_DIR / "parameters"
NOTEBOOK_DIR = PROJECT_ROOT / "02_notebooks"
OUTPUT_DIR = PROJECT_ROOT / "03_outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
PAPER_FIGURE_DIR = PROJECT_ROOT / "04_paper_figures"
DOCS_DIR = PROJECT_ROOT / "05_docs"
SRC_DIR = PROJECT_ROOT / "src"

for directory in [PROCESSED_DIR, FIGURE_DIR, TABLE_DIR, PAPER_FIGURE_DIR, DOCS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Parameter directory: {PARAMETER_DIR.relative_to(PROJECT_ROOT)}")
print(f"Output table directory: {TABLE_DIR.relative_to(PROJECT_ROOT)}")
print(f"Output figure directory: {FIGURE_DIR.relative_to(PROJECT_ROOT)}")


## Input schema definitions


In [ ]:
BASELINE_COLUMNS = ["parameter", "value", "unit", "description"]
RISK_COLUMNS = ["parameter", "value", "unit", "description"]
SCENARIO_COLUMNS = [
    "scenario",
    "description",
    "waste_growth_rate_per_year",
    "treatment_improvement_rate",
    "diversion_improvement_rate",
    "residual_restriction_rate",
]

BASELINE_PARAMETERS = [
    "initial_incoming_waste_tpd",
    "initial_treatment_capacity_tpd",
    "initial_residual_waste_tpd",
    "initial_diverted_waste_tpd",
    "organic_waste_tpd",
    "inorganic_waste_tpd",
    "initial_accumulated_waste_ton",
    "total_landfill_capacity_ton",
    "remaining_capacity_ton",
    "waste_pile_height_m",
    "time_step_day",
    "simulation_horizon_year",
]

RISK_PARAMETERS = [
    "risk_index_arithmetic",
    "risk_index_geometric",
    "risk_category",
    "bod",
    "cod",
    "tss",
    "ph",
    "capacity_operational_load_weight",
    "waste_characteristics_weight",
    "leachate_quality_weight",
]

SCENARIOS = [
    "S0_BAU",
    "S1_Increased_Waste_Generation",
    "S2_Improved_Organic_Treatment",
    "S3_Residual_Waste_Restriction",
    "S4_Combined_Intervention",
]


## Import simulator utilities


In [ ]:
from src.simulator_utils import (
    load_parameter_csv,
    parameters_to_dict,
    plot_bar_summary,
    plot_scenarios,
    run_sensitivity_analysis,
    run_simulation,
    summarize_scenarios,
    summarize_sensitivity,
    validate_required_columns,
    validate_required_parameters,
)


## 4. Load baseline parameters


In [ ]:
baseline_path = PARAMETER_DIR / "baseline_parameters.csv"
baseline_df = load_parameter_csv(
    baseline_path,
    required_columns=BASELINE_COLUMNS,
    required_parameters=BASELINE_PARAMETERS,
    parameter_column="parameter",
    file_label="baseline_parameters.csv",
)
baseline_parameters = parameters_to_dict(baseline_df)
baseline_df


## 5. Load risk parameters


In [ ]:
risk_path = PARAMETER_DIR / "risk_parameters.csv"
risk_df = load_parameter_csv(
    risk_path,
    required_columns=RISK_COLUMNS,
    required_parameters=RISK_PARAMETERS,
    parameter_column="parameter",
    file_label="risk_parameters.csv",
)
risk_parameters = parameters_to_dict(risk_df)
risk_df


## 6. Load scenario parameters


In [ ]:
scenario_path = PARAMETER_DIR / "scenario_parameters.csv"
scenario_df = load_parameter_csv(
    scenario_path,
    required_columns=SCENARIO_COLUMNS,
    required_parameters=SCENARIOS,
    parameter_column="scenario",
    file_label="scenario_parameters.csv",
)
scenario_df


## 7. Validate input data


In [ ]:
validate_required_columns(baseline_df, BASELINE_COLUMNS, "baseline_parameters.csv")
validate_required_parameters(baseline_df, BASELINE_PARAMETERS, "parameter", "baseline_parameters.csv")

validate_required_columns(risk_df, RISK_COLUMNS, "risk_parameters.csv")
validate_required_parameters(risk_df, RISK_PARAMETERS, "parameter", "risk_parameters.csv")

validate_required_columns(scenario_df, SCENARIO_COLUMNS, "scenario_parameters.csv")
validate_required_parameters(scenario_df, SCENARIOS, "scenario", "scenario_parameters.csv")

numeric_baseline = [
    "initial_incoming_waste_tpd",
    "initial_treatment_capacity_tpd",
    "initial_diverted_waste_tpd",
    "initial_accumulated_waste_ton",
    "total_landfill_capacity_ton",
    "time_step_day",
    "simulation_horizon_year",
]
for parameter in numeric_baseline:
    value = baseline_parameters[parameter]
    if not isinstance(value, (int, float, np.integer, np.floating)) or pd.isna(value):
        raise ValueError(f"Baseline parameter '{parameter}' must be numeric.")

rate_columns = [
    "waste_growth_rate_per_year",
    "treatment_improvement_rate",
    "diversion_improvement_rate",
    "residual_restriction_rate",
]
for column in rate_columns:
    scenario_df[column] = pd.to_numeric(scenario_df[column], errors="raise")

print("Input validation complete.")
print(f"Validated scenarios: {', '.join(scenario_df['scenario'].astype(str))}")


## 8. Define System Dynamics simulation function

The notebook uses `run_simulation()` from `src/simulator_utils.py`. The function implements the required equations for residual waste, Net Landfill Load, accumulated waste, remaining capacity, overload accumulation, and time to overload.


In [ ]:
simulation_function = run_simulation
print(simulation_function.__name__)


## 9. Run all scenarios


In [ ]:
scenario_results = []
for _, scenario in scenario_df.iterrows():
    result = run_simulation(
        baseline_parameters,
        scenario,
        scenario_name=str(scenario["scenario"]),
    )
    scenario_results.append(result)

daily_results = pd.concat(scenario_results, ignore_index=True)
daily_results.head()


## 10. Generate scenario summary table


In [ ]:
scenario_summary = summarize_scenarios(daily_results)
scenario_summary


## 11. Export results to CSV and Excel


In [ ]:
baseline_checked_path = TABLE_DIR / "baseline_parameters_checked.csv"
risk_checked_path = TABLE_DIR / "risk_parameters_checked.csv"
scenario_settings_path = TABLE_DIR / "scenario_settings.csv"
daily_results_path = TABLE_DIR / "simulation_daily_results.csv"
scenario_summary_path = TABLE_DIR / "scenario_summary.csv"

baseline_df.to_csv(baseline_checked_path, index=False)
risk_df.to_csv(risk_checked_path, index=False)
scenario_df.to_csv(scenario_settings_path, index=False)
daily_results.to_csv(daily_results_path, index=False)
scenario_summary.to_csv(scenario_summary_path, index=False)

print("Core result tables exported:")
for path in [
    baseline_checked_path,
    risk_checked_path,
    scenario_settings_path,
    daily_results_path,
    scenario_summary_path,
]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")
print("The final Excel workbook is written after sensitivity analysis sheets are created.")


## 12. Generate line charts


In [ ]:
time_series_plots = [
    (
        "net_landfill_load_tpd",
        "Net Landfill Load Forecast",
        "Net Landfill Load (ton/day)",
        "fig_nll_forecast.png",
    ),
    (
        "accumulated_waste_ton",
        "Accumulated Waste Projection",
        "Accumulated Waste (ton)",
        "fig_accumulated_waste_projection.png",
    ),
    (
        "remaining_capacity_ton",
        "Remaining Capacity Projection",
        "Remaining Capacity (ton)",
        "fig_remaining_capacity_projection.png",
    ),
    (
        "overload_accumulation_ton",
        "Overload Accumulation Projection",
        "Overload Accumulation (ton)",
        "fig_overload_accumulation_projection.png",
    ),
]

for y_column, title, ylabel, filename in time_series_plots:
    fig, ax = plot_scenarios(
        daily_results,
        y_column=y_column,
        title=title,
        ylabel=ylabel,
        output_path=FIGURE_DIR / filename,
    )
    plt.show()


## 13. Generate bar charts


In [ ]:
fig, ax = plot_bar_summary(
    scenario_summary,
    x_column="scenario",
    y_column="final_net_landfill_load_tpd",
    title="Final Net Landfill Load by Scenario",
    ylabel="Final Net Landfill Load (ton/day)",
    output_path=FIGURE_DIR / "fig_final_nll_by_scenario.png",
)
plt.show()

fig, ax = plot_bar_summary(
    scenario_summary,
    x_column="scenario",
    y_column="final_overload_accumulation_ton",
    title="Final Overload Accumulation by Scenario",
    ylabel="Final Overload Accumulation (ton)",
    output_path=FIGURE_DIR / "fig_final_overload_by_scenario.png",
)
plt.show()


## 14. Run sensitivity analysis


In [ ]:
base_scenario = scenario_df.loc[scenario_df["scenario"] == "S0_BAU"].iloc[0]
sensitivity_results = run_sensitivity_analysis(baseline_parameters, base_scenario)
sensitivity_results


## 15. Generate sensitivity summary table


In [ ]:
sensitivity_summary = summarize_sensitivity(sensitivity_results)

sensitivity_analysis_path = TABLE_DIR / "sensitivity_analysis.csv"
sensitivity_summary_path = TABLE_DIR / "sensitivity_summary.csv"
excel_output_path = TABLE_DIR / "system_dynamics_simulation_results.xlsx"

sensitivity_results.to_csv(sensitivity_analysis_path, index=False)
sensitivity_summary.to_csv(sensitivity_summary_path, index=False)

try:
    import openpyxl
except ImportError as exc:
    raise ImportError(
        "openpyxl is required to export the Excel workbook. Install dependencies with requirements.txt."
    ) from exc

with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    baseline_df.to_excel(writer, sheet_name="baseline_parameters", index=False)
    risk_df.to_excel(writer, sheet_name="risk_parameters", index=False)
    scenario_df.to_excel(writer, sheet_name="scenario_settings", index=False)
    daily_results.to_excel(writer, sheet_name="daily_results", index=False)
    scenario_summary.to_excel(writer, sheet_name="scenario_summary", index=False)
    sensitivity_results.to_excel(writer, sheet_name="sensitivity_analysis", index=False)
    sensitivity_summary.to_excel(writer, sheet_name="sensitivity_summary", index=False)

print("Sensitivity tables and final Excel workbook exported:")
for path in [sensitivity_analysis_path, sensitivity_summary_path, excel_output_path]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

sensitivity_summary


## 16. Generate sensitivity bar chart


In [ ]:
fig, ax = plot_bar_summary(
    sensitivity_summary,
    x_column="tested_parameter",
    y_column="range_overload_accumulation_ton",
    title="Sensitivity of Overload Accumulation",
    ylabel="Range of Final Overload Accumulation (ton)",
    output_path=FIGURE_DIR / "fig_sensitivity_overload_range.png",
)
plt.show()


## 17. Generate interpretation helper text for Results and Discussion


In [ ]:
best_overload = scenario_summary.sort_values("final_overload_accumulation_ton").iloc[0]
worst_overload = scenario_summary.sort_values("final_overload_accumulation_ton", ascending=False).iloc[0]
dominant_sensitivity = sensitivity_summary.iloc[0]

risk_category = risk_parameters.get("risk_category", "not specified")
risk_index = risk_parameters.get("risk_index_arithmetic", np.nan)

interpretation_text = f"""
Results and Discussion Helper

The Fuzzy AHP layer indicates a risk category of {risk_category} with an arithmetic risk index of {risk_index}. In this simulator, the risk layer supports interpretation and decision prioritization, while Net Landfill Load is calculated from operational waste flows.

Across the five scenarios, {best_overload['scenario']} produces the lowest final overload accumulation at {best_overload['final_overload_accumulation_ton']:,.2f} ton. The highest final overload accumulation occurs under {worst_overload['scenario']} at {worst_overload['final_overload_accumulation_ton']:,.2f} ton.

The largest one-at-a-time sensitivity effect on final overload accumulation is associated with {dominant_sensitivity['tested_parameter']}, with an overload range of {dominant_sensitivity['range_overload_accumulation_ton']:,.2f} ton across tested values.

Because the baseline remaining capacity is reported at or near zero, overload accumulation should be interpreted as a central planning indicator alongside final Net Landfill Load and accumulated waste.
""".strip()

print(interpretation_text)


## 18. Final checklist


In [ ]:
required_output_files = [
    TABLE_DIR / "simulation_daily_results.csv",
    TABLE_DIR / "scenario_summary.csv",
    TABLE_DIR / "scenario_settings.csv",
    TABLE_DIR / "baseline_parameters_checked.csv",
    TABLE_DIR / "risk_parameters_checked.csv",
    TABLE_DIR / "sensitivity_analysis.csv",
    TABLE_DIR / "sensitivity_summary.csv",
    TABLE_DIR / "system_dynamics_simulation_results.xlsx",
    FIGURE_DIR / "fig_nll_forecast.png",
    FIGURE_DIR / "fig_accumulated_waste_projection.png",
    FIGURE_DIR / "fig_remaining_capacity_projection.png",
    FIGURE_DIR / "fig_overload_accumulation_projection.png",
    FIGURE_DIR / "fig_final_nll_by_scenario.png",
    FIGURE_DIR / "fig_final_overload_by_scenario.png",
    FIGURE_DIR / "fig_sensitivity_overload_range.png",
]

missing_outputs = [path for path in required_output_files if not path.exists()]
if missing_outputs:
    missing_text = "\n".join(f"- {path.relative_to(PROJECT_ROOT)}" for path in missing_outputs)
    raise FileNotFoundError(f"Missing expected output file(s):\n{missing_text}")

print("Final checklist passed.")
print(f"Daily simulation rows: {len(daily_results):,}")
print(f"Scenario summary rows: {len(scenario_summary):,}")
print(f"Sensitivity runs: {len(sensitivity_results):,}")
if nbformat is not None:
    print(f"nbformat available: {nbformat.__version__}")
else:
    print("nbformat is not installed in this runtime; install requirements.txt for notebook editing workflows.")
